# 26.04.2026 analysis of TD2 output (A375_merged_transcriptome2.fas)

In [197]:
setwd("C:/My_documents/Documents/work/nikolas/oxford_nanopore/")

In [175]:
td2_res <- read.table("TD2_output/A375_merged_transcriptome2.fas.TD2.bed", skip = 1, header = F, stringsAsFactors = F)

In [176]:
head(td2_res)

                   V1 V2   V3
1       MSTRG.13427.4  0 5254
2 ENST00000445118.6_1  0 6606
3 ENST00000536743.5_1  0 1873
4        MSTRG.1710.1  0 2652
5 A375_batch_385.46.1  0 2127
6       MSTRG.15609.1  0 2144
                                                                                                    V4
1           ID=MSTRG.13427.4.p1;type:5prime_partial;len:51;gc:the;standard;code;MSTRG.13427.4:1-153(+)
2 ID=ENST00000445118.6_1.p9;type:complete;len:25;gc:the;standard;code;ENST00000445118.6_1:1852-1926(+)
3 ID=ENST00000536743.5_1.p7;type:complete;len:70;gc:the;standard;code;ENST00000536743.5_1:1622-1831(+)
4                  ID=MSTRG.1710.1.p5;type:complete;len:8;gc:the;standard;code;MSTRG.1710.1:475-498(+)
5 ID=A375_batch_385.46.1.p7;type:complete;len:12;gc:the;standard;code;A375_batch_385.46.1:1747-1782(+)
6            ID=MSTRG.15609.1.p10;type:complete;len:17;gc:the;standard;code;MSTRG.15609.1:1380-1430(+)
   V5 V6   V7   V8 V9 V10  V11 V12
1 501  +    1  153  0   1 5254   0

In [177]:
dim(td2_res)

[1] 654244     12

In [178]:
td2_res <- td2_res[!grepl('A375_batch', td2_res$V1), c('V1', 'V7', 'V8')]

In [179]:
colnames(td2_res) <- c('transcript_id', 'start', 'end')
td2_res$len_aa <- (td2_res$end - td2_res$start +1)/3
print(dim(td2_res))

[1] 626544      4


In [180]:
td2_res$type <- ifelse(grepl("MSTRG.", td2_res$transcript_id), "novel", "uniprot")

In [181]:
print(head(td2_res))

        transcript_id start  end len_aa    type
1       MSTRG.13427.4     1  153     51   novel
2 ENST00000445118.6_1  1852 1926     25 uniprot
3 ENST00000536743.5_1  1622 1831     70 uniprot
4        MSTRG.1710.1   475  498      8   novel
6       MSTRG.15609.1  1380 1430     17   novel
7        MSTRG.4634.3  1690 1737     16   novel


In [182]:
head(summary(as.factor(td2_res$transcript_id)))

ENST00000611082.1_1 ENST00000360280.7_1       MSTRG.13433.2        MSTRG.3622.6       MSTRG.15821.1 
                 96                  92                  89                  88                  84 
ENST00000616704.4_1 
                 82 

In [183]:
transcripts_table <- read.csv("A375_transcripts_table.tsv", header = T, stringsAsFactors = F, sep="\t")

In [184]:
transcripts_table$class2 <- transcripts_table$class_code
transcripts_table$class2[which(transcripts_table$class2 !='=')] <- "chimeric"
transcripts_table$class2[which(transcripts_table$class2 =='=')] <- "known"

In [185]:
transcripts_table <- transcripts_table[,c("ref_id", "class2")]

In [186]:
print(dim(transcripts_table))
head(transcripts_table)

[1] 104574      2


               ref_id   class2
1         MSTRG.291.1 chimeric
2         MSTRG.292.1 chimeric
3         MSTRG.293.1    known
4         MSTRG.293.2    known
5 ENST00000374266.6_2    known
6 ENST00000270812.6_2    known

In [187]:
head(summary(as.factor(transcripts_table$ref_id)))

            -   MSTRG.814.3  MSTRG.5157.1 MSTRG.8983.11  MSTRG.9582.1 MSTRG.10576.4 
         4776            20            17            17            17            16 

In [188]:
transcripts_table[transcripts_table$ref_id == "MSTRG.293.1",]

       ref_id class2
3 MSTRG.293.1  known

In [189]:
length(unique(transcripts_table$ref_id))

[1] 68740

In [190]:
library(dplyr)
df_transcripts = transcripts_table %>% 
  group_by(ref_id) %>% 
  summarise(most_frequent = names(which.max(table(class2))))

In [191]:
df_transcripts

# A tibble: 68,740 × 2
   ref_id               most_frequent
   <chr>                <chr>        
 1 -                    chimeric     
 2 ENST00000000233.9_1  chimeric     
 3 ENST00000000412.3    known        
 4 ENST00000001008.5_1  chimeric     
 5 ENST00000001146.6_1  chimeric     
 6 ENST00000002125.8_1  known        
 7 ENST00000002596.5_1  known        
 8 ENST00000003583.12_1 known        
 9 ENST00000003912.7_2  chimeric     
10 ENST00000005180.8_1  known        
# ℹ 68,730 more rows
# ℹ Use `print(n = ...)` to see more rows

In [192]:
summary(as.factor(df_transcripts$most_frequent))

chimeric    known 
   19387    49353 

In [193]:
#merging the two tables
td2_res$class <- df_transcripts$most_frequent[match(td2_res$transcript_id, df_transcripts$ref_id)]

In [194]:
head(td2_res)

        transcript_id start  end len_aa    type    class
1       MSTRG.13427.4     1  153     51   novel    known
2 ENST00000445118.6_1  1852 1926     25 uniprot    known
3 ENST00000536743.5_1  1622 1831     70 uniprot chimeric
4        MSTRG.1710.1   475  498      8   novel    known
6       MSTRG.15609.1  1380 1430     17   novel chimeric
7        MSTRG.4634.3  1690 1737     16   novel chimeric

In [169]:
write.csv(td2_res, "TD2_res_summary.csv", row.names = T)

In [195]:
summary(as.factor(td2_res$type))

  novel uniprot 
 307204  319340 

In [196]:
print(summary(as.factor(td2_res[which(td2_res$type=="novel"),"class"])))
print(summary(as.factor(td2_res[which(td2_res$type=="uniprot"),"class"])))

chimeric    known 
  122246   184958 
chimeric    known 
  121438   197902 


In [126]:
library(ggplot2)

In [173]:
options(repr.plot.width = 5, repr.plot.height = 3)

for (type in c("novel", "uniprot"))
  {
  print(paste("Type:", type))
  df_subset <- td2_res[td2_res$type == type,]
  df_num.per.tr <- df_subset %>% group_by(transcript_id) %>% summarise(count = n())
  g1 <- ggplot(df_num.per.tr, aes(x = count)) + labs(title = paste("Distribution of number of ORFs per transcript for", type, "transcripts"), x='number of ORFs', y='Frequency') + 
  geom_bar()
  #print(g1)
  ggsave(paste0(type,"_number_orf_per.transcript.pdf"), plot = g1, width = 8, height = 6)
  write.csv(df_num.per.tr, paste0(type,"_number_orf_per.transcript.csv"), row.names = F)

  df_num.per.len <- df_subset %>% group_by(len_aa) %>% summarise(count = n())
  g2 <- ggplot(df_num.per.len, aes(x = len_aa, y=count)) + labs(title = paste("Distribution of number of ORFs per AA length for", type, "transcripts"), x='ORF length, AA', y='number of ORFs') +
  geom_point() #+ ylim(0, 10000) 
  #print(g2)
  ggsave(paste0(type,"_number_orf_per.AA.length.pdf"), plot = g2, width = 8, height = 6)
  write.csv(df_num.per.len, paste0(type,"_number_orf_per.AA.length.csv"), row.names = F)

  df_num.per.tr2 <- df_subset %>% group_by(transcript_id, class) %>% summarise(count = n()) %>% group_by(class) %>% summarise(mean_count = mean(count))
  print(df_num.per.tr2)

  df_avg.len <- df_subset %>% group_by(class) %>% summarise(avg_len = mean(len_aa))
  print(df_avg.len)
}

[1] "Type: novel"
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by transcript_id and class.
ℹ Output is grouped by transcript_id.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(transcript_id, class))` for ]8;;x-r-help:dplyr::dplyr_byper-operation grouping]8;; instead.
# A tibble: 2 × 2
  class    mean_count
  <chr>         <dbl>
1 chimeric       15.1
2 known          10.8
# A tibble: 2 × 2
  class    avg_len
  <chr>      <dbl>
1 chimeric    48.8
2 known       67.4
[1] "Type: uniprot"
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by transcript_id and class.
ℹ Output is grouped by transcript_id.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(transcript_id, class))` for ]8;;x-r-help:dplyr::dplyr_byper-operation grouping]8;; instead.
# A tibble: 2 × 2
  class    mean_count
  <chr>         <dbl>
1 chimeric      11.0 
2 known          6.24
# 

# 29.04.2026 Summarizing immunopeptidomics results

In [199]:
setwd("C:/My_documents/Documents/work/nikolas/oxford_nanopore/")

In [207]:
imm_df <- read.csv("immunopeptidomics/Limma_results_V1_all-comparisons.csv", stringsAsFactors = F)
print(dim(imm_df))
imm_df <- imm_df[grepl("MSTRG.", imm_df$PG.ProteinGroups),]
print(dim(imm_df))

[1] 147828     64
[1] 90102    64


In [208]:
head(imm_df)

    X
2   2
3   3
4   4
5   5
10 10
12 12
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  PG.ProteinGroups
2                                                                                                                                                                                                                                                                                                                                                                                                                          

In [209]:
summary(as.factor(imm_df$method))

CV_-50 CV_-65 
 68142  21960 

In [210]:
write.csv(imm_df, "immunopeptidomics/Limma_results_V1_all-comparisons_MSTRG_only.csv", row.names = F)

In [217]:
imm_df_subset <- imm_df[(imm_df$logFC > 2) & (imm_df$fdr.limma < 0.05),]
dim(imm_df_subset)

[1] 1881   64

In [218]:
summary(as.factor(imm_df_subset$method))

CV_-50 CV_-65 
  1522    359 

In [219]:
write.csv(imm_df_subset, "immunopeptidomics/Limma_results_V1_all-comparisons_MSTRG_only_logFC_FDR_filtered.csv", row.names = F)